# Pandas Basics

Same questions as `sql_basics.ipynb`, answered with Pandas DataFrames instead of SQL.

---

## Datasets

- `surah.json`: Surah details and revelation place.
- `ayah.json`: Verse text and word counts.
- `juz.json`: Juz divisions and verse counts.
- `sajda.json`: Prostration verses.
- `matching-ayah.json`: Nested verse similarity matches.

## Load the Data

Read each JSON file into a DataFrame using `pandas.read_json`.

In [19]:
import pandas as pd
import pyarrow

surah = pd.read_json("quran-data/surah.json")
ayah = pd.read_json("quran-data/ayah.json")

juz = pd.read_json("quran-data/juz.json")

sajda = pd.read_json("quran-data/sajda.json")
surah.head()

,id,name_arabic,name_english,name_transliteration,verses_count,revelation_place,revelation_order,pages
0,1,الفاتحة,Al-Fatihah,Al-Fātiĥah,7,Makkah,5,"[1, 1]"
1,2,البقرة,Al-Baqarah,Al-Baqarah,286,Madinah,87,"[2, 49]"
2,3,آل عمران,Ali 'Imran,Āli `Imrān,200,Madinah,89,"[50, 76]"
3,4,النساء,An-Nisa,An-Nisā,176,Madinah,92,"[77, 106]"
4,5,المائدة,Al-Ma'idah,Al-Mā'idah,120,Madinah,112,"[106, 127]"


### Question 1: Surahs and Verses by Revelation Place

Count total surahs and total verses for Makkah vs Madinah, including each place's percentage of the total.

**Pandas Concepts:** `groupby`, `agg`, computing a percentage against the column total.

In [32]:
# groupby + agg calculates the average verses per Surah in each group.
summary_by_place = surah.groupby("revelation_place").agg(
    avg_verse_count=("verses_count", "mean"),
    surah_count=("id", "count"),
).sort_values("surah_count", ascending=False)

summary_by_place

,avg_verse_count,surah_count
revelation_place,,
Makkah,53.639535,86
Madinah,57.964286,28


In [20]:
# groupby + agg gives one row per revelation place with both totals.
surah_stats = surah.groupby("revelation_place").agg(
    total_surahs=("id", "count"),
    total_verses=("verses_count", "sum"),
)

# Dividing by the column sum mirrors the SQL window function SUM(...) OVER ().
surah_stats["surah_percentage"] = (100.0 * surah_stats["total_surahs"] / surah_stats["total_surahs"].sum()).round(2)
surah_stats["verse_percentage"] = (100.0 * surah_stats["total_verses"] / surah_stats["total_verses"].sum()).round(2)

surah_stats = surah_stats.sort_values("total_surahs", ascending=False).reset_index()
surah_stats[["revelation_place", "total_surahs", "surah_percentage", "total_verses", "verse_percentage"]]

,revelation_place,total_surahs,surah_percentage,total_verses,verse_percentage
0,Makkah,86,75.44,4613,73.97
1,Madinah,28,24.56,1623,26.03


### Question 2: Surahs with Prostration Verses (Sajdah)

List Sajdah verses with their Surah names and Arabic text.

**Pandas Concepts:** `merge` joins related rows; `sort_values` sorts the final results.

In [21]:
# merge adds verse details when both verse keys match.
sajda_verses = sajda.merge(ayah, on="verse_key")
# merge adds Surah names when the Surah numbers match.
sajda_verses = sajda_verses.merge(surah, left_on="surah_number", right_on="id", suffixes=("", "_surah"))

sajda_verses = sajda_verses.rename(
    columns={"name_arabic": "surah_name", "name_english": "surah_name_english", "text": "verse_text"}
)

sajda_verses = sajda_verses.sort_values("sajdah_number")
sajda_verses[["sajdah_number", "verse_key", "sajdah_type", "surah_name", "surah_name_english", "verse_text"]]

,sajdah_number,verse_key,sajdah_type,surah_name,surah_name_english,verse_text
0,1,7:206,optional,الأعراف,Al-A'raf,إِنَّ ٱلَّذِينَ عِندَ رَبِّكَ لَا يَسۡتَكۡبِرُ...
1,2,13:15,optional,الرعد,Ar-Ra'd,وَلِلَّهِۤ يَسۡجُدُۤ مَن فِي ٱلسَّمَٰوَٰتِ وَٱ...
2,3,16:50,optional,النحل,An-Nahl,يَخَافُونَ رَبَّهُم مِّن فَوۡقِهِمۡ وَيَفۡعَلُ...
3,4,17:109,optional,الإسراء,Al-Isra,وَيَخِرُّونَ لِلۡأَذۡقَانِ يَبۡكُونَ وَيَزِيدُ...
4,5,19:58,optional,مريم,Maryam,أُوْلَٰٓئِكَ ٱلَّذِينَ أَنۡعَمَ ٱللَّهُ عَلَيۡ...
5,6,22:18,optional,الحج,Al-Hajj,أَلَمۡ تَرَ أَنَّ ٱللَّهَ يَسۡجُدُۤ لَهُۥۤ مَن...
6,7,25:60,optional,الفرقان,Al-Furqan,وَإِذَا قِيلَ لَهُمُ ٱسۡجُدُواْۤ لِلرَّحۡمَٰنِ...
7,8,27:26,optional,النمل,An-Naml,ٱللَّهُ لَآ إِلَٰهَ إِلَّا هُوَ رَبُّ ٱلۡعَرۡ...
8,9,32:15,required,السجدة,As-Sajdah,إِنَّمَا يُؤۡمِنُ بِـَٔايَٰتِنَا ٱلَّذِينَ إِذ...
9,10,38:24,optional,ص,Sad,قَالَ لَقَدۡ ظَلَمَكَ بِسُؤَالِ نَعۡجَتِكَ إِل...


### Question 3: Top 5 Longest Juz by Verse Count

Find the 5 Juz with the most verses, including the names and text of their starting and ending ayahs.

**Pandas Concepts:** `sort_values` + `head` for the top rows; `str.split` extracts parts of a verse key; `merge` combines related rows; `to_parquet` saves a DataFrame.

In [22]:
# sort_values + head keeps only the five largest Juz.
top_juz = juz.sort_values("verses_count", ascending=False).head(5)
top_juz[["juz_number", "verses_count", "first_verse_key", "last_verse_key"]]

,juz_number,verses_count,first_verse_key,last_verse_key
29,30,564,78:1,114:6
28,29,431,67:1,77:50
26,27,399,51:31,57:29
22,23,357,36:28,39:31
18,19,339,25:21,27:55


In [33]:
top_juz = top_juz.copy()

# str.split gets the Surah and ayah numbers from keys such as '2:255'.
top_juz["start_surah"] = top_juz["first_verse_key"].str.split(":").str[0].astype(int)
top_juz["start_aya"] = top_juz["first_verse_key"].str.split(":").str[1].astype(int)
top_juz["end_surah"] = top_juz["last_verse_key"].str.split(":").str[0].astype(int)
top_juz["end_aya"] = top_juz["last_verse_key"].str.split(":").str[1].astype(int)

# Create lookup tables (i.e., dictionaries) to be used by .map().
# map() uses these lookups to find:
# - the text of each ayah from its verse_key
# - the Arabic name of each Surah from its ID
# set_index() turns a column into lookup keys.
verse_text = ayah.set_index("verse_key")["text"]
surah_name = surah.set_index("id")["name_arabic"]

top_juz["start_aya_text"] = top_juz["first_verse_key"].map(verse_text)
top_juz["end_aya_text"] = top_juz["last_verse_key"].map(verse_text)
top_juz["start_surah_name"] = top_juz["start_surah"].map(surah_name)
top_juz["end_surah_name"] = top_juz["end_surah"].map(surah_name)

result = top_juz.sort_values("verses_count", ascending=False)[[
    "juz_number", "verses_count",
    "start_surah", "start_surah_name", "start_aya", "start_aya_text",
    "end_surah", "end_surah_name", "end_aya", "end_aya_text",
]]

display(result)

,juz_number,verses_count,start_surah,start_surah_name,start_aya,start_aya_text,end_surah,end_surah_name,end_aya,end_aya_text
29,30,564,78,النبإ,1,عَمَّ يَتَسَآءَلُونَ,114,الناس,6,مِنَ ٱلۡجِنَّةِ وَٱلنَّاسِ
28,29,431,67,الملك,1,تَبَٰرَكَ ٱلَّذِي بِيَدِهِ ٱلۡمُلۡكُ وَهُوَ عَ...,77,المرسلات,50,فَبِأَيِّ حَدِيثِۭ بَعۡدَهُۥ يُؤۡمِنُونَ
26,27,399,51,الذاريات,31,۞ قَالَ فَمَا خَطۡبُكُمۡ أَيُّهَا ٱلۡمُرۡسَلُونَ,57,الحديد,29,لِّئَلَّا يَعۡلَمَ أَهۡلُ ٱلۡكِتَٰبِ أَلَّا يَ...
22,23,357,36,يس,28,۞ وَمَآ أَنزَلۡنَا عَلَىٰ قَوۡمِهِۦ مِنۢ بَعۡ...,39,الزمر,31,ثُمَّ إِنَّكُمۡ يَوۡمَ ٱلۡقِيَٰمَةِ عِندَ رَبّ...
18,19,339,25,الفرقان,21,۞ وَقَالَ ٱلَّذِينَ لَا يَرۡجُونَ لِقَآءَنَا ...,27,النمل,55,أَئِنَّكُمۡ لَتَأۡتُونَ ٱلرِّجَالَ شَهۡوَةٗ مّ...


### Question 4: Top 5 Surahs by Total Word Count

Find the 5 largest Surahs by total word count.

**Pandas Concepts:** `merge` matches each Surah to its ayahs; `groupby` + `sum` totals the words; `sort_values` + `head` returns the top five.

In [27]:
# merge connects each Surah to its ayahs using the Surah number.
surah_words = surah.merge(ayah, left_on="id", right_on="surah_number")

# groupby + sum adds the word counts for all ayahs in each Surah.
top_surahs = surah_words.groupby(["name_arabic", "name_english", "revelation_place"]).agg(
    total_words=("words_count", "sum")
).reset_index()

top_surahs = top_surahs.rename(columns={"name_arabic": "surah_name", "name_english": "surah_name_english"})
top_surahs.sort_values("total_words", ascending=False).head(5)

,surah_name,surah_name_english,revelation_place,total_words
15,البقرة,Al-Baqarah,Madinah,6117
91,النساء,An-Nisa,Madinah,3747
0,آل عمران,Ali 'Imran,Madinah,3481
4,الأعراف,Al-A'raf,Makkah,3320
7,الأنعام,Al-An'am,Makkah,3050


### Question 5: Longest Verses

Find the 10 verses with the most words and characters.

**Pandas Concepts:** `str.len` counts characters; `sort_values` + `head` restricts the number of rows.

In [29]:
longest_verses = ayah.copy()

# str.len counts the characters in each verse.
longest_verses["character_count"] = longest_verses["text"].str.len()

longest_verses = longest_verses.sort_values("words_count", ascending=False).head(10)
longest_verses[["verse_key", "words_count", "character_count", "text"]]

,verse_key,words_count,character_count,text
288,2:282,128,1188,يَٰٓأَيُّهَا ٱلَّذِينَ ءَامَنُوٓاْ إِذَا تَدَا...
504,4:12,88,681,۞ وَلَكُمۡ نِصۡفُ مَا تَرَكَ أَزۡوَٰجُكُمۡ إِن...
2821,24:31,78,758,وَقُل لِّلۡمُؤۡمِنَٰتِ يَغۡضُضۡنَ مِنۡ أَبۡصَٰ...
5494,73:20,78,704,۞ إِنَّ رَبَّكَ يَعۡلَمُ أَنَّكَ تَقُومُ أَدۡن...
2851,24:61,76,670,لَّيۡسَ عَلَى ٱلۡأَعۡمَىٰ حَرَجٞ وَلَا عَلَى ٱ...
446,3:154,75,631,ثُمَّ أَنزَلَ عَلَيۡكُم مِّنۢ بَعۡدِ ٱلۡغَمِّ ...
108,2:102,74,662,وَٱتَّبَعُواْ مَا تَتۡلُواْ ٱلشَّيَٰطِينُ عَلَ...
202,2:196,73,633,وَأَتِمُّواْ ٱلۡحَجَّ وَٱلۡعُمۡرَةَ لِلَّهِۚ ف...
503,4:11,71,602,يُوصِيكُمُ ٱللَّهُ فِيٓ أَوۡلَٰدِكُمۡۖ لِلذَّك...
3585,33:53,69,625,يَٰٓأَيُّهَا ٱلَّذِينَ ءَامَنُواْ لَا تَدۡخُلُ...
